# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# For visualization
import matplotlib.pyplot as plt
%matplotlib inline

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate record sets, their fields, and example fields for further exploration.


In [ ]:
# List all available record sets with their @id
print("Available Record Sets (by @id and name):")
for record_set in dataset.metadata.record_sets:
    print(f"- @id: {record_set.id}\n  name: {record_set.name}")
    # For each record set, show field @id and name
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}, name: {field.name}, datatype: {field.data_type}")
    print()

# Save record set @ids for later usage
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Preview a few sample records for the first record set
if record_set_ids:
    sample_records = list(dataset.records(record_set=record_set_ids[0]))
    print(f"Sample records from '{record_set_ids[0]}':")
    for row in sample_records[:2]:
        print(row)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Below, we extract all data as pandas DataFrames, keyed by their record set `@id`.

Replace `<your_desired_record_set_id>` with one of the `@id` values listed above to choose the main data table.


In [ ]:
# Extract tables: Load each record set into a dataframe
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Choose the main record set for exploration (edit as needed)
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"Fields (columns) in DataFrame for record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalization, and grouping.

We'll pick a numeric field for illustration. Replace `<numeric_field_id>` and `<group_field_id>` with actual `@id` values available in the chosen table.

In [ ]:
# Select numeric and group fields by @id.
# You can find the exact @ids from the above Data Overview section and the DataFrame columns.

df = dataframes[main_record_set_id]
# Example: find numeric columns (float/int)
numeric_columns = df.select_dtypes(include=['float', 'int']).columns.tolist()
if numeric_columns:
    numeric_field_id = numeric_columns[0]
    threshold = df[numeric_field_id].mean()  # Example threshold at mean value
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by a non-numeric attribute if available
    group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object']
    if group_candidates:
        group_field_id = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field_id}':")
        print(grouped_df.head())
    else:
        print("No string group field found for grouping.")
else:
    print("No numeric columns available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field, and if grouped data is available, plot group means.

In [ ]:
# Plot a histogram of the numeric field
if 'numeric_field_id' in locals():
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If we have a grouped_df from above, plot group means
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        grouped_df.plot(kind='bar', x=group_field_id, y=numeric_field_id, legend=False)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()


## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² dataset using `mlcroissant`. We:

- Loaded dataset metadata and reviewed available record sets and fields (all by `@id`).
- Extracted data from the selected record set into pandas DataFrames.
- Applied basic filtering, normalization, and grouping to a numeric field, referencing columns by their `@id`.
- Visualized the numeric data distribution and mean value by group.

Adapt this template to perform further analyses and visualizations as desired, always referencing record sets and fields using their Croissant `@id` for reproducibility.